RAG Pipelines- Data Ingestion to Vector DB 

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ts181\AppData\Local\Temp\ipykernel_24308\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./data/pdf")

Found 1 PDF files to process

Processing: book1.pdf
  ✓ Loaded 22 pages

Total documents loaded: 22


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-03-09T15:07:25+05:30', 'moddate': '2026-04-24T12:28:49+05:30', 'trapped': '/False', 'source': 'data\\pdf\\book1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '62', 'source_file': 'book1.pdf', 'file_type': 'pdf'}, page_content='‘मेरी आवाज़ ही पहचान है!’  अपनी आवाज़ से भारत ही न हीं बल््ककि ल्वश्व किे  किोन े-किोन े में \nअपनी पहचान बनाने वाली ‘भारत रत्न’ लता मंगेशकिर स ंगीत किे  संसार किा वह नाम हैं ल्िनकिे  \nल्वषय में यल्ि बात न हो तो भारतीय स ंगीत किी बात अधूरी रह िाएगी। इंिौर, मध्यप्रिेश में िन्मी \nलता मंगेशकिर ने मात्र पाँच व ष्ष किी आयु में अपन े ल्पता से संगीत किी प्रारंल्भकि ल्शक्ा ली और \nिीवनपययंत संगीत किे  प्रल्त समल्प्षत रहीं। \nिीवन किे  अनेकि उतार-चढ़ाव व स ंघषषों किे  होत े हुए भी लता मंगेशकिर ने ल्किस प्रकिार संगीत \nऔर पररवार किे  प्रल्त अपन े उत्तरिाल्यत्व किो ल्नभाया, आइए पढ़त े हैं उनकिे  साथ साक्ात्किार में...\nयतीींद्र मिश्र\nयतीींद्र मिश्

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 22 documents into 57 chunks

Example chunk:
Content: ‘मेरी आवाज़ ही पहचान है!’  अपनी आवाज़ से भारत ही न हीं बल््ककि ल्वश्व किे  किोन े-किोन े में 
अपनी पहचान बनाने वाली ‘भारत रत्न’ लता मंगेशकिर स ंगीत किे  संसार किा वह नाम हैं ल्िनकिे  
ल्वषय में यल्ि ब...
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-03-09T15:07:25+05:30', 'moddate': '2026-04-24T12:28:49+05:30', 'trapped': '/False', 'source': 'data\\pdf\\book1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '62', 'source_file': 'book1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-03-09T15:07:25+05:30', 'moddate': '2026-04-24T12:28:49+05:30', 'trapped': '/False', 'source': 'data\\pdf\\book1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '62', 'source_file': 'book1.pdf', 'file_type': 'pdf'}, page_content='‘मेरी आवाज़ ही पहचान है!’  अपनी आवाज़ से भारत ही न हीं बल््ककि ल्वश्व किे  किोन े-किोन े में \nअपनी पहचान बनाने वाली ‘भारत रत्न’ लता मंगेशकिर स ंगीत किे  संसार किा वह नाम हैं ल्िनकिे  \nल्वषय में यल्ि बात न हो तो भारतीय स ंगीत किी बात अधूरी रह िाएगी। इंिौर, मध्यप्रिेश में िन्मी \nलता मंगेशकिर ने मात्र पाँच व ष्ष किी आयु में अपन े ल्पता से संगीत किी प्रारंल्भकि ल्शक्ा ली और \nिीवनपययंत संगीत किे  प्रल्त समल्प्षत रहीं। \nिीवन किे  अनेकि उतार-चढ़ाव व स ंघषषों किे  होत े हुए भी लता मंगेशकिर ने ल्किस प्रकिार संगीत \nऔर पररवार किे  प्रल्त अपन े उत्तरिाल्यत्व किो ल्नभाया, आइए पढ़त े हैं उनकिे  साथ साक्ात्किार में...\nयतीींद्र मिश्र\nयतीींद्र मिश्

Embedding and Vector DB

In [7]:
###Embedding and Vector DB
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\ts181\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
###Embedding part
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            dimension = self.model.get_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2284.06it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\ts181\AppData\Local\Temp\ipykernel_24308\558370891.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [18]:
##vectore store
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
chunks


[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-03-09T15:07:25+05:30', 'moddate': '2026-04-24T12:28:49+05:30', 'trapped': '/False', 'source': 'data\\pdf\\book1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '62', 'source_file': 'book1.pdf', 'file_type': 'pdf'}, page_content='‘मेरी आवाज़ ही पहचान है!’  अपनी आवाज़ से भारत ही न हीं बल््ककि ल्वश्व किे  किोन े-किोन े में \nअपनी पहचान बनाने वाली ‘भारत रत्न’ लता मंगेशकिर स ंगीत किे  संसार किा वह नाम हैं ल्िनकिे  \nल्वषय में यल्ि बात न हो तो भारतीय स ंगीत किी बात अधूरी रह िाएगी। इंिौर, मध्यप्रिेश में िन्मी \nलता मंगेशकिर ने मात्र पाँच व ष्ष किी आयु में अपन े ल्पता से संगीत किी प्रारंल्भकि ल्शक्ा ली और \nिीवनपययंत संगीत किे  प्रल्त समल्प्षत रहीं। \nिीवन किे  अनेकि उतार-चढ़ाव व स ंघषषों किे  होत े हुए भी लता मंगेशकिर ने ल्किस प्रकिार संगीत \nऔर पररवार किे  प्रल्त अपन े उत्तरिाल्यत्व किो ल्नभाया, आइए पढ़त े हैं उनकिे  साथ साक्ात्किार में...\nयतीींद्र मिश्र\nयतीींद्र मिश्

In [20]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-03-09T15:07:25+05:30', 'moddate': '2026-04-24T12:28:49+05:30', 'trapped': '/False', 'source': 'data\\pdf\\book1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '62', 'source_file': 'book1.pdf', 'file_type': 'pdf'}, page_content='‘मेरी आवाज़ ही पहचान है!’  अपनी आवाज़ से भारत ही न हीं बल््ककि ल्वश्व किे  किोन े-किोन े में \nअपनी पहचान बनाने वाली ‘भारत रत्न’ लता मंगेशकिर स ंगीत किे  संसार किा वह नाम हैं ल्िनकिे  \nल्वषय में यल्ि बात न हो तो भारतीय स ंगीत किी बात अधूरी रह िाएगी। इंिौर, मध्यप्रिेश में िन्मी \nलता मंगेशकिर ने मात्र पाँच व ष्ष किी आयु में अपन े ल्पता से संगीत किी प्रारंल्भकि ल्शक्ा ली और \nिीवनपययंत संगीत किे  प्रल्त समल्प्षत रहीं। \nिीवन किे  अनेकि उतार-चढ़ाव व स ंघषषों किे  होत े हुए भी लता मंगेशकिर ने ल्किस प्रकिार संगीत \nऔर पररवार किे  प्रल्त अपन े उत्तरिाल्यत्व किो ल्नभाया, आइए पढ़त े हैं उनकिे  साथ साक्ात्किार में...\nयतीींद्र मिश्र\nयतीींद्र मिश्

In [21]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]

### generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

###store embeddings in vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 57 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.06s/it]

Generated embeddings with shape: (57, 384)
Adding 57 documents to vector store...
Successfully added 57 documents to vector store
Total documents in collection: 57


Retriever Pipeline From VectorStore

In [22]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [24]:
rag_retriever

In [26]:
rag_retriever.retrieve("ऐसी भी बातीें होतीी हैं")


Retrieving documents for query: 'ऐसी भी बातीें होतीी हैं'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.52it/s]

Generated embeddings with shape: (1, 384)


Retrieved 1 documents (after filtering)


[{'id': 'doc_69ce4af4_47',
  'content': 'कीमजए और बतीाइए मक साइबर सुरक्ा मनयिों का प्रयोग करतीे हुए इस प्रकार की धोखाधड़ी से \nमकस प्रकार बिा जा सकतीा है?\n https://cybercrime.gov.in/UploadMedia/CyberSafetyHindi.Pdf\n04 Lata Mangeshker.indd   7904 Lata Mangeshker.indd   79 09-Mar-26   3:07:34 PM09-Mar-26   3:07:34 PM',
  'metadata': {'source': 'data\\pdf\\book1.pdf',
   'creationdate': '2026-03-09T15:07:25+05:30',
   'file_type': 'pdf',
   'content_length': 274,
   'doc_index': 47,
   'page': 17,
   'total_pages': 22,
   'moddate': '2026-04-24T12:28:49+05:30',
   'producer': 'Adobe PDF Library 18.0',
   'source_file': 'book1.pdf',
   'trapped': '/False',
   'creator': 'Adobe InDesign 21.2 (Windows)',
   'page_label': '79'},
  'similarity_score': 0.006549715995788574,
  'distance': 0.9934502840042114,
  'rank': 1}]

Integration vectorDb context pipleine with LLM output

In [40]:
# simple rag pipeline with GROQ LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

##initialize the GROQ LLM(set your GROQ_API_KEY in environment)
groq_api_key =  os.getenv("GROQ_API_KEY")


llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [44]:
answer=rag_simple("who is lata mangeshkar?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'who is lata mangeshkar?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Lata Mangeshkar (1929 – 2022) was a legendary Indian playback singer, often called the “Nightingale of India.” She recorded thousands of songs in Hindi and many other Indian languages, becoming one of the most celebrated and influential vocalists in Indian cinema.
